In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as si
from scipy.stats import norm
import warnings
warnings.filterwarnings("ignore")
import QuantLib as ql
import plotly.graph_objects as go
import os
import regex as re
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
from scipy import interpolate
from scipy.optimize import minimize
import seaborn as sns
from tqdm import tqdm
from datetime import datetime, timedelta

In [ ]:
#main code
import math
import numpy as np  # Import numpy for mathematical functions

class TreeNode:
    def __init__(self, stock_price):
        self.stockPrice = stock_price
        self.option_value = 0.0  # Added for option valuation

    def add_dividend_present_values(self, deltaT, risk_free_rate, dividends, dividend_dates):
        for dividend, date in zip(dividends, dividend_dates):
            present_value = self.dividend_present_value(dividend, date, risk_free_rate, deltaT)
            self.stockPrice += present_value

    def dividend_present_value(self, dividend, date, risk_free_rate, deltaT):
        # Assuming exponential discounting for the dividend present value
        return dividend * math.exp(-risk_free_rate * (date / 365.0))

class TreeNodeFactory:
    def __init__(self):
        self.nodes = {}

    def reset_nodes(self):
        self.nodes = {}

    def create_node(self, j_step, i, u, d, stock_price, deltaT):
        # Calculate the adjusted price based on the up and down factors
        adjusted_price = stock_price * (u ** (j_step - i)) * (d ** i)
        node_key = (j_step, i)
        if node_key not in self.nodes:  # Check if the node does not already exist
            self.nodes[node_key] = TreeNode(adjusted_price)

    def get_node(self, j_step, i):
        node_key = (j_step, i)
        return self.nodes.get(node_key)  # Retrieve the node using the same key structure as in create_node

class OptionPricer:
    def __init__(self, steps, deltaT):
        self.steps = steps
        self.deltaT = deltaT
        self.myFactory = TreeNodeFactory()

    def set_parameters(self, stock_price, risk_free_rate, sigma, dividends, dividend_dates):
        self.stockPrice = stock_price
        self.riskFreeRate = risk_free_rate
        self.sigma = sigma
        self.dividends = dividends
        self.dividend_dates = dividend_dates
        self.u = np.exp((risk_free_rate - 0) * deltaT + sigma * np.sqrt(deltaT))
        self.d = np.exp((risk_free_rate - 0) * deltaT - sigma * np.sqrt(deltaT))

    def get_dividend_adjusted_underlying(self):
        pv = sum([self.present_value(div, date) for div, date in zip(self.dividends, self.dividend_dates)])
        return self.stockPrice - pv

    def present_value(self, dividend, date):
        return dividend * math.exp(-self.riskFreeRate * (date / 365.0))

    def create_stock_price_tree(self):
        self.stockPrice = self.get_dividend_adjusted_underlying()
        self.myFactory.reset_nodes()
        for j_step in range(self.steps + 1):
            for i in range(j_step + 1):
                self.myFactory.create_node(j_step, i, self.u, self.d, self.stockPrice, self.deltaT)
        for j_step in range(self.steps + 1):
            for i in range(j_step + 1):
                node = self.myFactory.get_node(j_step, i)
                node.add_dividend_present_values(self.deltaT, self.riskFreeRate, self.dividends, self.dividend_dates)

    def price_american_option(self, option_type, strike_price):
        self.create_stock_price_tree()  # Ensure tree is built with current parameters
        r = self.riskFreeRate
        q = 0  # Simplification: no continuous dividend yield
        p_up = (math.exp((r - q) * self.deltaT) - self.d) / (self.u - self.d)
        p_down = 1 - p_up

        for i in range(self.steps + 1):
            node = self.myFactory.get_node(self.steps, i)
            if option_type == 'put':
                node.option_value = max(strike_price - node.stockPrice, 0)
            elif option_type == 'call':
                node.option_value = max(node.stockPrice - strike_price, 0)

        for j_step in range(self.steps - 1, -1, -1):
            for i in range(j_step + 1):
                node = self.myFactory.get_node(j_step, i)
                up_node = self.myFactory.get_node(j_step + 1, i)
                down_node = self.myFactory.get_node(j_step + 1, i + 1)
                expected_option_value = math.exp(-r * self.deltaT) * (p_up * up_node.option_value + p_down * down_node.option_value)
                
                if option_type == 'put':
                    exercise_value = max(strike_price - node.stockPrice, 0)
                elif option_type == 'call':
                    exercise_value = max(node.stockPrice - strike_price, 0)

                node.option_value = max(expected_option_value, exercise_value)

        return self.myFactory.get_node(0, 0).option_value

In [ ]:
# Example usage:
steps = 100
stock_price = 100
deltaT = 1/365
risk_free_rate = 0.05
dividends = [1, 2, 1.5, 0.5]
dividend_dates = [30, 60, 90, 120]
sigma = 0.2  # Volatility of the stock
option_type = 'put'  # Change to 'call' for a call option
strike_price = 100

option_pricer = OptionPricer(steps, stock_price, deltaT, risk_free_rate, dividends, dividend_dates, sigma)
option_pricer.create_stock_price_tree()
option_price = option_pricer.price_american_option(option_type, strike_price)
print(f"American {option_type} option price: {option_price}")

In [ ]:
#import data and combining
tmp_q = pd.read_csv("fdm_price_1_q_1.csv")
tmp_div = pd.read_csv("fdm_price_1_div_amt_1.csv")
tmp = pd.read_csv("fdm_price_1_1.csv")

In [ ]:
data = pd.concat([tmp, tmp_q, tmp_div], axis=1)

# If you need to merge them based on a common column (e.g., 'ID', 'Date', etc.)
# Replace 'key' with the name of the column that is common to all three dataframes
# data = pd.merge(tmp, tmp_q, on='key')
# data = pd.merge(data, tmp_div, on='key')

# Display the combined dataframe
print(data)

In [ ]:
data1 = data.iloc[:10000]
data1

In [ ]:
# usage

# Initialize the new columns for put and call prices
data1['put_price'] = pd.Series(dtype=float)
data1['call_price'] = pd.Series(dtype=float)

# Iterate through each row in data1
for index, row in data1.iterrows():
    pricer = OptionPricer(steps=100, deltaT=1/365)
    pricer.set_parameters(
        stock_price=1, 
        risk_free_rate=row['r'], 
        sigma=row['iv'],
        dividends=[row['div_amt']], 
        dividend_dates=[row['time_to_first_div[day]']]
    )

    # Price the American options
    price_put = pricer.price_american_option(option_type='put', strike_price=row['strike'])
    price_call = pricer.price_american_option(option_type='call', strike_price=row['strike'])

    # Assign the results back to the data1 DataFrame
    data1.at[index, 'put_price'] = price_put
    data1.at[index, 'call_price'] = price_call

# Now data1 includes the original data plus the new put and call prices
print(data1.head())


# results = []
# for index, row in data1.iterrows():
#     pricer = OptionPricer(steps=100, deltaT=1/365)
#     pricer.set_parameters(stock_price=1, risk_free_rate=row['r'], sigma=row['iv'],
#                           dividends=[row['div_amt']], dividend_dates=[row['time_to_first_div[day]']])
#     price_put = pricer.price_american_option(option_type='put', strike_price=row['strike'])
#     price_call = pricer.price_american_option(option_type='call', strike_price=row['strike'])
#     results.append({'index': index, 'put_price': price_put, 'call_price': price_call})

# # Convert results to DataFrame for better visualization and analysis
# results_df = pd.DataFrame(results)
# print(results_df.head())
#row['stock_price']

In [ ]:
data1.to_csv('newresults.csv', index=False)

In [ ]:
# Ensure 'data1' includes all necessary columns: 'strike', 'call_price', and 'amer_call'
# Assuming 'amer_call' is a column in your original DataFrame that contains the American Call prices

plt.figure(figsize=(10, 6))
plt.scatter(data1['strike'], data1['call_price'], label='Calculated Call Price', marker='o')
plt.scatter(data1['strike'], data1['amer_call'], label='American Call Price', marker='x')

plt.title('Comparison of Calculated Call Prices and American Call Prices')
plt.xlabel('Strike Price')
plt.ylabel('Price')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Ensure 'data1' includes all necessary columns: 'strike', 'put_price', and 'amer_put'
# Assuming 'amer_put' is a column in your original DataFrame that contains the American Put prices

plt.figure(figsize=(10, 6))
plt.scatter(data1['strike'], data1['put_price'], label='Calculated Put Price', marker='o')
plt.scatter(data1['strike'], data1['amer_put'], label='American Put Price', marker='x')

plt.title('Comparison of Calculated Put Prices and American Put Prices')
plt.xlabel('Strike Price')
plt.ylabel('Price')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Update the filter to include T[day] = 730, div_amt = 0.0175, and time_to_first_div[day] = 1
data730_div_day1 = data1[(data1['T[day]'] == 730) & (data1['div_amt'] == 0.0175) & (data1['time_to_first_div[day]'] == 1)]

# Calculate the differences for the specifically filtered data
data730_div_day1['call_price_diff'] = data730_div_day1['call_price'] - data730_div_day1['amer_call']
data730_div_day1['put_price_diff'] = data730_div_day1['put_price'] - data730_div_day1['amer_put']

# Plot for call prices difference with specific conditions
plt.figure(figsize=(10, 6))
plt.scatter(data730_div_day1['strike'], data730_div_day1['call_price_diff'], label='Difference in Call Prices', marker='o')
plt.title('Call Price Differences: T[day]=730, div_amt=0.175, div_day=1')
plt.xlabel('Strike Price')
plt.ylabel('Price Difference')
plt.legend()
plt.grid(True)
plt.show()

# Plot for put prices difference with specific conditions
plt.figure(figsize=(10, 6))
plt.scatter(data730_div_day1['strike'], data730_div_day1['put_price_diff'], label='Difference in Put Prices', marker='o')
plt.title('Put Price Differences: T[day]=730, div_amt=0.175, div_day=1')
plt.xlabel('Strike Price')
plt.ylabel('Price Difference')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Compute the difference between calculated call prices and American call prices
results_df['price_difference'] = results_df['call_price'] - results_df['amer_call']

# Now plot the difference
plt.figure(figsize=(10, 6))
plt.plot(results_df['strike'], results_df['price_difference'], label='Call Price Difference', marker='o', color='red')

plt.title('Difference Between Calculated and American Call Prices')
plt.xlabel('Strike Price')
plt.ylabel('Price Difference')
plt.axhline(0, color='black', linewidth=0.5)  # Adds a horizontal line at zero difference
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Compute the difference between calculated put prices and American put prices
results_df['put_price_difference'] = results_df['put_price'] - data1['amer_put']

# Now plot the difference for put options
plt.figure(figsize=(10, 6))
plt.plot(results_df['strike'], results_df['put_price_difference'], label='Put Price Difference', marker='o', color='blue')

plt.title('Difference Between Calculated and American Put Prices')
plt.xlabel('Strike Price')
plt.ylabel('Price Difference')
plt.axhline(0, color='black', linewidth=0.5)  # Adds a horizontal line at zero difference
plt.legend()
plt.grid(True)
plt.show()